In [1]:
import sqlite3
import random
from typing import List, Dict
# from transformers import PreTrainedTokenizerBase
# import torch

import sys
from pathlib import Path

# Add parent folder to search path to import from config.py
parent_dir = Path().resolve().parent
print(parent_dir)
sys.path.append(str(parent_dir))

from config import Settings


SYSTEM_PROMPT = (
    "You explain concepts to people at different age levels. "
    "Adapt explanations to the user (e.g. age, education, job)."
)



/home/schmi/projects/explain2me


In [2]:

# --------------------------------------------------
# Fetch data from DB
# --------------------------------------------------


def fetch_training_data(db_path: str):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        SELECT d.page_id, p.title, d.kind, d.content
        FROM definitions d
        JOIN pages p ON d.page_id = p.id
        WHERE d.kind IN ('simple', 'technical', 'kids')
    """)

    rows = cur.fetchall()
    conn.close()

    return rows

db_path = Settings.get_db_path()
test_fetch = fetch_training_data(db_path=db_path)

test_fetch[1]


(19,
 'Analytics',
 'simple',
 '[{"heading": "Introduction", "paragraphs": ["Analytics is the systematic processing of data or statistics. This means discovering, studying and explaining important patterns in data.", "Analytics turns raw data into useful information for making better decisions. Analytics uses the application of statistics, computer programming, and operations research to gain information from meanings of data."]}]')

In [3]:

# --------------------------------------------------
# Audience generator based on kind
# --------------------------------------------------

def generate_user_prompt(title: str, kind: str) -> str:
    """
    Generate user request based on definition kind.
    """

    if kind == "kids":
        age = random.randint(6, 13)
        templates = [
            f"Can you explain to me what {title} is? I am {age} years old.",
            f"What is {title}? Please explain it for a {age}-year-old.",
            f"I'm {age}. Can you help me understand {title}?",
            f"I am in elementary school. What is {title}?",
            f"Can you explain {title} in very basic terms?"
        ]

    elif kind == "simple":
        ages = random.randint(12, 18)
        templates = [
            f"Explain {title} to a {ages}-year-old student.",
            f"What is {title}? I'm in high school.",
            f"Can you explain {title} in simple terms?"
        ]

    elif kind == "technical":
        roles = [
            "PhD student",
            "graduate student",
            "researcher",
            "engineer",
            "domain expert"
        ]
        role = random.choice(roles)

        templates = [
            f"What is {title}? Explain it to a {role}.",
            f"Provide a detailed explanation of {title} suitable for a {role}.",
            f"I am a {role}. Give me a technical explanation of {title}."
        ]

    else:
        templates = [f"Explain {title}."]

    return random.choice(templates)


test_gen_user_prompt = generate_user_prompt(title='correlation', kind='technical')
test_gen_user_prompt


'What is correlation? Explain it to a engineer.'

In [9]:

# --------------------------------------------------
# LoRA Adapter: training dataset builder
# --------------------------------------------------

def build_lora_training_dataset(
    db_path: str,
) -> List[Dict]:

    rows = fetch_training_data(db_path)

    dataset = []

    for page_id, title, kind, content in rows:

        user_prompt = generate_user_prompt(title, kind)

        assistant_content = f"{title}:\n\n{content}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_content},
        ]

        dataset.append({
            "page_id": page_id,
            "messages": messages,
        })

    return dataset


test_lora_training_data = build_lora_training_dataset(db_path=db_path)
len(test_lora_training_data), test_lora_training_data

(347,
 [{'page_id': 9,
   'messages': [{'role': 'system',
     'content': 'You explain concepts to people at different age levels. Adapt explanations to the user (e.g. age, education, job).'},
    {'role': 'user',
     'content': 'What is Blockchain? Explain it to a domain expert.'},
    {'role': 'assistant',
     'content': 'Blockchain:\n\n[{"heading": "Introduction", "paragraphs": ["A blockchain is a distributed ledger with growing lists of records ( blocks ) that are securely linked together via cryptographic hashes. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data (generally represented as a Merkle tree, where data nodes are represented by leaves). Since each block contains information about the previous block, they effectively form a chain ( viz. linked list data structure), with each additional block linking to the ones before it. Consequently, blockchain transactions are resistant to alteration because, once recorded, the data in 

In [10]:
[x.get('messages') for x in test_lora_training_data]

[[{'role': 'system',
   'content': 'You explain concepts to people at different age levels. Adapt explanations to the user (e.g. age, education, job).'},
  {'role': 'user',
   'content': 'What is Blockchain? Explain it to a domain expert.'},
  {'role': 'assistant',
   'content': 'Blockchain:\n\n[{"heading": "Introduction", "paragraphs": ["A blockchain is a distributed ledger with growing lists of records ( blocks ) that are securely linked together via cryptographic hashes. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data (generally represented as a Merkle tree, where data nodes are represented by leaves). Since each block contains information about the previous block, they effectively form a chain ( viz. linked list data structure), with each additional block linking to the ones before it. Consequently, blockchain transactions are resistant to alteration because, once recorded, the data in any given block cannot be changed retroactively

In [12]:
# Load simple wiki page data

import json

training_data_path = "training_data.json"

    
def save_data(data, data_path):
	with open(data_path, "w", encoding="utf-8") as f:
		json.dump(data, f, indent=2, ensure_ascii=False)

def load_data(data_path):
	with open(data_path,"r") as file:
		test_training_data = json.load(file)
	return test_training_data

save_data(data=test_lora_training_data,
		  data_path=training_data_path)

test_training_data = load_data(data_path=training_data_path)
    
test_training_data[:2]

[{'page_id': 9,
  'messages': [{'role': 'system',
    'content': 'You explain concepts to people at different age levels. Adapt explanations to the user (e.g. age, education, job).'},
   {'role': 'user',
    'content': 'What is Blockchain? Explain it to a domain expert.'},
   {'role': 'assistant',
    'content': 'Blockchain:\n\n[{"heading": "Introduction", "paragraphs": ["A blockchain is a distributed ledger with growing lists of records ( blocks ) that are securely linked together via cryptographic hashes. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data (generally represented as a Merkle tree, where data nodes are represented by leaves). Since each block contains information about the previous block, they effectively form a chain ( viz. linked list data structure), with each additional block linking to the ones before it. Consequently, blockchain transactions are resistant to alteration because, once recorded, the data in any given blo

In [ ]:

# from transformers import AutoModelForCausalLM, AutoTokenizer

# # Set device
# device = "cuda" if torch.cuda.is_available() else "cpu"

# Settings.MAX_INPUT_TOKENS
# model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id).to(
#     device
# )

# tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

In [ ]:
# toy_messages = [{"role": 'system', 'content': 'sys content 1'},
#                 {"role": 'user', 'content': 'sys content 1'},
#                 {"role": 'assistant', 'content': 'sys content 2'*5 + 'lets add additional words so that it  should at least exceed it'}]
# toy_messages


# trucated_tokens = tokenizer.apply_chat_template(
#     toy_messages,
#     tokenize=True,
#     add_generation_prompt=False,
#     max_length = Settings.MAX_INPUT_TOKENS,
#     truncation=True,
# )

# untruncated_tokens = tokenizer.apply_chat_template(
#     toy_messages,
#     tokenize=True,
#     add_generation_prompt=False,
# )

# print("Final token length:", len(untruncated_tokens['input_ids']))
# print(len(trucated_tokens['input_ids']))
# print("Max allowed:", Settings.MAX_INPUT_TOKENS)

